# 02: Running the Agent

In Notebook 01 we explored the dataset and tools. This notebook shows how to run the
`KnowledgeGroundedAgent` in practice.

## What You'll Learn

1. The agent's PlanReAct architecture and the `AgentResponse` data structure
2. Running a question with live progress display
3. Inspecting the response: plan, tool calls, sources, and reasoning
4. Multi-turn conversations using session state
5. Observability with Langfuse tracing

## Prerequisites

Complete Notebook 01. You'll need `GOOGLE_API_KEY` in your `.env` file.
For tracing (Section 4): `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` are also required.

In [1]:
import os
import uuid
from pathlib import Path

from aieng.agent_evals.knowledge_qa import KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from aieng.agent_evals.langfuse import init_tracing
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


# Set working directory to the repository root
if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent)
    print(f"Working directory set to: {Path('').absolute()}")


load_dotenv(verbose=True)
console = Console(width=100)

Working directory set to: /home/coder/eval-agents/implementations


## 1. Agent Architecture

The `KnowledgeGroundedAgent` is built on Google ADK and combines two patterns:

**PlanReAct** — Before executing, the agent produces an explicit research plan with numbered
steps. Each step has a type (`SEARCH`, `FETCH`, `ANALYZE`) and a status that transitions from
`pending` → `in_progress` → `completed` (or `failed`/`skipped`). The plan can be revised
mid-run if the agent encounters unexpected results.

**ReAct loop** — Within each step, the agent alternates between reasoning (Thought), acting
(tool call), and observing (tool response).

### The `AgentResponse` Object

After running, `agent.answer_async(question)` returns an `AgentResponse`:

| Field | Type | Description |
|-------|------|-------------|
| `text` | `str` | The final answer |
| `plan` | `ResearchPlan` | Numbered steps with statuses |
| `tool_calls` | `list[dict]` | Every tool invocation during execution |
| `sources` | `list[GroundingChunk]` | URLs used as evidence |
| `reasoning_chain` | `list[str]` | The model's intermediate reasoning |
| `total_duration_ms` | `int` | Wall-clock execution time |

In [2]:
agent = KnowledgeGroundedAgent(enable_planning=True)

config_table = Table(title="Agent Configuration", show_header=False)
config_table.add_column("Setting", style="cyan")
config_table.add_column("Value", style="white")
config_table.add_row("Model", agent.model)
config_table.add_row("Planning", "PlanReAct (enabled)")
config_table.add_row("Session Service", "InMemorySessionService")
config_table.add_row("Tools", "google_search, web_fetch, fetch_file, grep_file, read_file")

console.print(config_table)

                              Agent Configuration                               
┌─────────────────┬────────────────────────────────────────────────────────────┐
│ Model           │ gemini-2.5-flash                                           │
│ Planning        │ PlanReAct (enabled)                                        │
│ Session Service │ InMemorySessionService                                     │
│ Tools           │ google_search, web_fetch, fetch_file, grep_file, read_file │
└─────────────────┴────────────────────────────────────────────────────────────┘

## 2. Running a Question

`run_with_display` executes the agent in a Jupyter notebook with a live progress display showing:

- The research plan with step statuses (updating in real time)
- Tool calls as they fire

We'll use a question that requires web search — the agent must find and verify a specific fact,
not recall it from training data.

In [3]:
question = "Any news regarding earning calls from Canada's Big Five? Summarize if there is any."

response = await run_with_display(agent, question)

In [4]:
display_response(
    console,
    response.text,
    subtitle=(
        f"Duration: {response.total_duration_ms / 1000:.1f}s  |  "
        f"Tool calls: {len(response.tool_calls)}  |  "
        f"Sources: {len(response.sources)}"
    ),
)

╭───────────────────────────────────────────── Answer ─────────────────────────────────────────────╮
│ There are no earning calls scheduled for Canada's Big Five banks on April 21, 2026. The most     │
│ recent earnings calls for these banks were for their First Quarter (Q1) 2026 results, which were │
│ reported in late February 2026. Their next earnings calls for Q2 2026 are generally scheduled    │
│ for late May 2026.                                                                               │
│                                                                                                  │
│ Here is a summary of the Q1 2026 earnings for Canada's Big Five banks:                           │
│                                                                                                  │
│ Royal Bank of Canada (RBC)                                                                       │
│                                                                                                  │
│  • Reported Net Income: $5.79 billion (up from $5.13 billion in Q1 2025).                        │
│  • Reported Diluted EPS: $4.03 (up from $3.54 in Q1 2025).                                       │
│  • Adjusted Diluted EPS: $4.08 (up from $3.62 in Q1 2025).                                       │
│  • Dividend Changes: Information not found in the retrieved source.                              │
│  • Key Drivers: Record results driven by strong earnings growth, robust balance sheet, and       │
│    capital position, with strength in personal banking, commercial banking, wealth management    │
│    (due to higher fee-based client assets from market appreciation and net sales), and capital   │
│    markets. Provision for credit losses increased to $1.09 billion from $1.05 billion a year     │
│    earlier.                                                                                      │
│  • Source: BNN Bloomberg — 2026-02-26 — Royal Bank of Canada's profit rises on capital markets,  │
│    personal banking strength                                                                     │
│                                                                                                  │
│ Toronto-Dominion Bank (TD)                                                                       │
│                                                                                                  │
│  • Reported Net Income: $4,043 million (up 45% year-over-year from $2,793 million in Q1 2025).   │
│  • Adjusted Net Income: $4,216 million (up 16% year-over-year from $3,623 million in Q1 2025).   │
│  • Reported Diluted EPS: $2.34 (up 50.97% year-over-year from $1.55 in Q1 2025).                 │
│  • Adjusted Diluted EPS: $2.44 (up 20.79% year-over-year from $2.02 in Q1 2025).                 │
│  • Dividend Changes: Dividends per share increased to $1.08 in Q1 2026, up from $1.05 in Q1 2025 │
│    and Q4 2025.                                                                                  │
│  • Key Drivers: Strong results with record adjusted earnings and significant year-over-year      │
│    adjusted return on equity growth, reflecting momentum across businesses. Achieved robust      │
│    trading and fee income growth in markets-driven businesses, volume growth in Canadian         │
│    Personal and Commercial Banking, and margin expansion.                                        │
│  • Source: TD Bank Group Reports First Quarter 2026 Results — 2026-02-26 —                       │
│    https://td.mediaroom.com/2026-02-26-TD-Bank-Group-Reports-First-Quarter-2026-Results          │
│                                                                                                  │
│ Bank of Nova Scotia (Scotiabank)                                                                 │
│                                                                                                  │
│  • Reported Net Income: $2,299 million (increased from $993

### 2.1 Inspecting the Response

The `AgentResponse` object contains the full execution trace.

In [5]:
plan = response.plan

plan_table = Table(title="Research Plan")
plan_table.add_column("#", style="cyan", width=3)
plan_table.add_column("Step", style="white")
plan_table.add_column("Type", style="dim", width=12)
plan_table.add_column("Status", style="green")

for step in plan.steps:
    icon = {"completed": "✓", "failed": "✗", "skipped": "○"}.get(step.status.value, "·")
    desc = step.description[:70] + "..." if len(step.description) > 70 else step.description
    plan_table.add_row(str(step.step_id), desc, step.step_type, f"{icon} {step.status.value}")

console.print(plan_table)

                                           Research Plan                                            
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ #   ┃ Step                                                          ┃ Type         ┃ Status      ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 1   │ Read the `sources.md` file to understand the credible         │ research     │ ✓ completed │
│     │ sources.                                                      │              │             │
│ 2   │ Search for news regarding earning calls from Canada's Big     │ research     │ ✓ completed │
│     │ Five banks f...                                               │              │             │
│ 3   │ If no results are found for the current date, broaden the     │ research     │ ✓ completed │
│     │ search to in...                                               │              │             │
│ 4   │ For each promising search result, fetch the web page content. │ research     │ ✓ completed │
│ 5   │ Extract and summarize the key information about earning calls │ research     │ ✓ completed │
│     │ for each...                                                   │              │             │
│ 6   │ Compile and present the summarized information, ensuring to   │ research     │ ✓ completed │
│     │ cite all s...                                                 │              │             │
│ 7   │ Read the `sources.md` file to understand the credible sources │ research     │ ✓ completed │
│     │ (already...                                                   │              │             │
│ 8   │ Read the `sources.md` file to understand the credible sources │ research     │ ✓ completed │
│     │ (already...                                                   │              │             │
│ 9   │ Search for news regarding earning calls from Canada's Big     │ research     │ ✓ completed │
│     │ Five banks f...                                               │              │             │
│ 10  │ If no results are found for the current date, broaden the     │ research     │ ○ skipped   │
│     │ search to in...                                               │              │             │
│ 11  │ For each promising search result, fetch the web page content  │ research     │ ○ skipped   │
│     │ (Scotiaba...                                                  │              │             │
│ 12  │ **Search for "Royal Bank of Canada Q1 2026 earnings"          │ research     │ ○ skipped   │
│     │ specifically targ...                                          │              │             │
│ 13  │ Fetch the most relevant article from a credible source and    │ research     │ ○ skipped   │
│     │ extract the...                                                │              │             │
│ 14  │ Compile and present the summarized information, ensuring to   │ research     │ ○ skipped   │
│     │ cite all s...                                                 │              │             │
└─────┴───────────────────────────────────────────────────────────────┴──────────────┴─────────────┘

In [6]:
if response.tool_calls:
    tools_table = Table(title="Tool Calls")
    tools_table.add_column("#", style="dim", width=3)
    tools_table.add_column("Tool", style="cyan", width=16)
    tools_table.add_column("Arguments (truncated)", style="white")

    for i, tc in enumerate(response.tool_calls[:15], 1):
        name = tc.get("name", "unknown")
        args = str(tc.get("args", {}))
        args = args[:70] + "..." if len(args) > 70 else args
        tools_table.add_row(str(i), name, args)

    if len(response.tool_calls) > 15:
        tools_table.add_row("...", f"({len(response.tool_calls) - 15} more)", "")

    console.print(tools_table)
else:
    console.print("[dim]No tool calls recorded[/dim]")

                                             Tool Calls                                             
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Tool             ┃ Arguments (truncated)                                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ read_file        │ {'file_path':                                                           │
│     │                  │ '/home/coder/eval-agents/aieng-eval-agents/aieng/agent_e...             │
│ 2   │ google_search    │ {'query': "Canada's Big Five banks earning calls April 21, 2026"}       │
│ 3   │ web_fetch        │ {'url':                                                                 │
│     │                  │ 'https://www.scotiabank.com/content/dam/scotiabank/corporate/q...       │
│ 4   │ web_fetch        │ {'query': 'Bank of Montreal (BMO) Q1 2026 net income, EPS,              │
│     │                  │ year-over-y...                                                          │
│ 5   │ web_fetch        │ {'query': 'CIBC Q1 2026 net income, EPS, year-over-year change,         │
│     │                  │ divide...                                                               │
│ 6   │ web_fetch        │ {'query': 'TD Bank Q1 2026 net income, EPS, year-over-year change,      │
│     │                  │ div...                                                                  │
│ 7   │ web_fetch        │ {'query': 'Royal Bank of Canada (RBC) Q1 2026 net income, EPS,          │
│     │                  │ year-ov...                                                              │
│ 8   │ web_fetch        │ {'query': 'Royal Bank of Canada (RBC) Q1 2026 reported and adjusted     │
│     │                  │ ne...                                                                   │
│ 9   │ google_search    │ {'query': 'Royal Bank of Canada Q1 2026 earnings Reuters or             │
│     │                  │ Bloomberg'...                                                           │
│ 10  │ web_fetch        │ {'url':                                                                 │
│     │                  │ 'https://www.bnnbloomberg.ca/business/company-news/2026/02/26/...       │
└─────┴──────────────────┴─────────────────────────────────────────────────────────────────────────┘

In [7]:
if response.sources:
    seen: set[str] = set()
    sources_table = Table(title="Sources")
    sources_table.add_column("#", style="dim", width=3)
    sources_table.add_column("URL", style="blue")

    for src in response.sources:
        if src.uri and src.uri not in seen:
            seen.add(src.uri)
            url = src.uri[:85] + "..." if len(src.uri) > 85 else src.uri
            sources_table.add_row(str(len(seen)), url)
        if len(seen) >= 10:
            break

    console.print(sources_table)
else:
    console.print("[dim]No sources recorded[/dim]")

                                             Sources                                              
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ URL                                                                                      ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ https://www.scotiabank.com/ca/en/about/investors-shareholders/financial-result.html      │
│ 2   │ https://www.scotiabank.com/content/dam/scotiabank/corporate/quarterly-reports/2026/q1... │
│ 3   │ https://ca.investing.com/news/company-news/bank-of-montreal-bmo-q1-2026-earnings-call... │
│ 4   │ https://www.prnewswire.com/news-releases/bmo-financial-group-reports-first-quarter-20... │
│ 5   │ https://seekingalpha.com/article/4874478-bank-of-montreal-2026-q1-results-earnings-ca... │
│ 6   │ https://www.investing.com/news/transcripts/earnings-call-transcript-national-bank-of-... │
│ 7   │ https://www.nbc.ca/about-us/news-media/press-release/2026/20260203-nbc-announcement-r... │
│ 8   │ https://www.cibc.com/content/dam/cibc-public-assets/about-cibc/investor-relations/pdf... │
│ 9   │ https://www.cibc.com/en/about-cibc/investor-relations/quarterly-results.html             │
│ 10  │ https://www.bnnbloomberg.ca/business/company-news/2026/02/26/cibc-reports-310b-first-... │
└─────┴──────────────────────────────────────────────────────────────────────────────────────────┘

## 3. Multi-Turn Conversations

The agent uses an `InMemorySessionService` to maintain conversation context across turns.
Pass the same `session_id` to link questions together — the agent will use prior context
when answering follow-up questions.

In [8]:
session_id = str(uuid.uuid4())
console.print(f"Session ID: [dim]{session_id}[/dim]\n")

# First turn: establish a subject
response1 = await agent.answer_async(
    "Any news regarding earning calls from Canada's Big Five? Summarize if there is any.",
    session_id=session_id,
)
display_response(console, response1.text, title="Turn 1")

# Second turn: follow-up that references the prior context
response2 = await agent.answer_async(
    "How is each bank's performance compared to that of last quarter or year?",
    session_id=session_id,
)
display_response(console, response2.text, title="Turn 2 (follow-up)")

Session ID: 0166425e-3150-47f7-b79a-d61c0b4c8721

2026-04-21 03:15:04,649 INFO aieng.agent_evals.knowledge_qa.agent: Answering question (type='', max_llm_calls=20, thinking_budget=8192): Any news regarding earning calls from Canada's Big Five? Summarize if there is any....
2026-04-21 03:15:04,784 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-04-21 03:15:08,248 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-04-21 03:15:08,250 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 4 steps
2026-04-21 03:15:08,251 INFO aieng.agent_evals.knowledge_qa.agent: Tool call: read_file({'file_path': '/home/coder/eval-agents/aieng-eval-agents/aieng/agent_evals/knowledge_qa/sources.md'})
2026-04-21 03:15:08,253 INFO aieng.agent_evals.knowledge_qa.agent: Tool response: read_file completed
2026-04-21 03:15:08,855 INFO google_adk.google.adk.models.gemini_context_cache_manager: Cache created successfully: c

╭───────────────────────────────────────────── Turn 1 ─────────────────────────────────────────────╮
│ As of April 21, 2026, Canada's Big Five banks do not have earnings calls scheduled for April     │
│ 2026. Their first-quarter 2026 earnings calls primarily took place in February 2026. The         │
│ second-quarter 2026 earnings reports and calls for these banks are anticipated in late May 2026. │
│                                                                                                  │
│ Here is the confirmed schedule for the Q2 2026 earnings calls:                                   │
│                                                                                                  │
│  • Toronto-Dominion Bank (TD): May 28, 2026 (TD.com — Investor Relations)                        │
│  • Bank of Nova Scotia (Scotiabank): May 27, 2026 (Newswire.ca — 2026-04-21 — Scotiabank         │
│    Comments on Expected Contribution from KeyCorp's First Quarter Earnings)                      │
│  • Bank of Montreal (BMO): May 27, 2026 (BMO.com — 2026-02-25 — BMO Financial Group Reports      │
│    First Quarter 2026 Results)                                                                   │
│  • Canadian Imperial Bank of Commerce (CIBC): May 28, 2026, at 7:30 AM ET (CIBC.com — Investor   │
│    Relations)                                                                                    │
│                                                                                                  │
│ For Royal Bank of Canada (RBC), the anticipated Q2 2026 earnings release date is May 28, 2026.   │
│ However, this date could not be definitively verified from RBC's official investor relations     │
│ page or other accessible credible sources at this time.                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-04-21 03:18:26,325 INFO aieng.agent_evals.knowledge_qa.agent: Answering question (type='', max_llm_calls=20, thinking_budget=8192): How is each bank's performance compared to that of last quarter or year?...
2026-04-21 03:18:26,450 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-04-21 03:18:26,452 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-04-21 03:18:29,426 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-04-21 03:18:29,429 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 4 steps
2026-04-21 03:18:29,430 INFO aieng.agent_evals.knowledge_qa.agent: Tool call: google_search({'query': 'RBC Q1 2026 earnings report net income EPS vs Q4 2025 vs Q1 2025'})
2026-04-21 03:18:29,537 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-04-21 03:18:44,451 INFO aieng.agent_evals.knowledge_qa.agent

╭─────────────────────────────────────── Turn 2 (follow-up) ───────────────────────────────────────╮
│ I need more information to answer your question precisely. Could you please specify:             │
│                                                                                                  │
│  1 Which banks are you interested in? (e.g., "Canada's Big Five banks," "specific US regional    │
│    banks," etc.)                                                                                 │
│  2 What specific performance metrics are you interested in? (e.g., net income, revenue, earnings │
│    per share, stock performance, etc.)                                                           │
│  3 For what period are you seeking this comparison? (e.g., "most recent quarter," "Q4 2025 vs Q4 │
│    2024," etc.)                                                                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

## 4. Observability with Langfuse

Langfuse captures a full trace of every agent run using OpenTelemetry, giving you visibility into:

- Every tool call and its arguments
- Every LLM call with prompts and completions
- Timing for each span
- The full agent execution tree

This is essential for debugging failures, measuring latency, and comparing configurations.

### Trace Structure

```
Trace: agent run
├── Span: planning (PlanReAct)
│   └── LLM Call: create_plan
├── Span: step-1-execution
│   ├── Tool Call: google_search
│   ├── Tool Call: web_fetch
│   └── LLM Call: step_summary
├── Span: step-2-execution
│   └── ...
└── Span: synthesis
    └── LLM Call: final_answer
```

### Prerequisites

Set these in your `.env` file:
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`
- `LANGFUSE_HOST` (optional, defaults to `https://cloud.langfuse.com`)

In [9]:
langfuse_configured = all(
    [
        os.getenv("LANGFUSE_PUBLIC_KEY"),
        os.getenv("LANGFUSE_SECRET_KEY"),
    ]
)

if langfuse_configured:
    console.print("[green]✓[/green] Langfuse credentials found")
else:
    console.print("[yellow]⚠[/yellow] Langfuse credentials not found — tracing cells will be skipped")
    console.print("[dim]Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY in .env[/dim]")

✓ Langfuse credentials found

In [10]:
tracing_enabled = init_tracing()

if tracing_enabled:
    console.print("[green]✓[/green] Langfuse tracing initialized")
else:
    console.print("[yellow]⚠[/yellow] Tracing not enabled (check credentials)")

2026-04-21 03:18:58,252 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed
2026-04-21 03:18:58,343 INFO aieng.agent_evals.langfuse: Langfuse tracing initialized successfully (endpoint: https://us.cloud.langfuse.com/api/public/otel)


✓ Langfuse tracing initialized

In [11]:
if tracing_enabled:
    from langfuse import Langfuse

    langfuse = Langfuse()
    traced_agent = KnowledgeGroundedAgent(enable_planning=True)
    traced_question = "What programming language was created by Guido van Rossum, and in what year?"

    console.print(Panel(traced_question, title="Traced Question", border_style="green"))

    with langfuse.start_as_current_span(name="knowledge-agent", input=traced_question):
        trace_id = langfuse.get_current_trace_id()
        traced_response = await traced_agent.answer_async(traced_question)
        langfuse.update_current_span(output=traced_response.text)

    display_response(
        console,
        traced_response.text,
        subtitle=f"Duration: {traced_response.total_duration_ms / 1000:.1f}s",
    )
else:
    console.print("[dim]Skipping (Langfuse not configured)[/dim]")
    trace_id = None

╭──────────────────────────────────────── Traced Question ─────────────────────────────────────────╮
│ What programming language was created by Guido van Rossum, and in what year?                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-04-21 03:18:58,526 INFO aieng.agent_evals.knowledge_qa.agent: Answering question (type='', max_llm_calls=20, thinking_budget=8192): What programming language was created by Guido van Rossum, and in what year?...
/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/adk/models/google_llm.py:172: UserWarning: [EXPERIMENTAL] GeminiContextCacheManager: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  cache_manager = GeminiContextCacheManager(self.api_client)
2026-04-21 03:18:58,639 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-04-21 03:18:59,567 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-04-21 03:18:59,572 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 2 steps
2026-04-21 03:18:59,572 INFO aieng.agent_evals.knowledge_qa.agen

╭───────────────────────────────────────────── Answer ─────────────────────────────────────────────╮
│ The programming language created by Guido van Rossum is Python, and it was first released in     │
│ 1991.                                                                                            │
╰───────────────────────────────────────── Duration: 5.5s ─────────────────────────────────────────╯

In [12]:
if tracing_enabled:
    from IPython.display import HTML, display  # noqa: A004
    from langfuse import Langfuse
    from opentelemetry import trace as otel_trace

    provider = otel_trace.get_tracer_provider()
    if hasattr(provider, "force_flush"):
        provider.force_flush(timeout_millis=5000)
    console.print("[green]✓[/green] Traces flushed to Langfuse")

    if trace_id:
        trace_url = Langfuse().get_trace_url(trace_id=trace_id)
        display(HTML(f'<p>View trace: <a href="{trace_url}" target="_blank">{trace_url}</a></p>'))

✓ Traces flushed to Langfuse

### 4.1 Viewing Traces in the Langfuse UI

Open your Langfuse project and navigate to **Traces**. Each run appears as a
tree of spans. Useful things to look at:

- **Span timeline** — which steps take the most time?
- **Tool call arguments** — what search queries did the agent use?
- **LLM interactions** — what did the model reason about before calling each tool?
- **Errors** — red spans show where failures occurred

You can also filter by trace name, time range, or input content.

## Summary

In this notebook you learned:

1. **Creating the agent** — `KnowledgeGroundedAgent(enable_planning=True)` with PlanReAct
2. **Running questions** — `run_with_display` for live notebook progress; `agent.answer_async` for raw access
3. **The `AgentResponse`** — plan, tool calls, sources, reasoning, and timing in one object
4. **Multi-turn conversations** — linking turns with `session_id`
5. **Langfuse tracing** — `init_tracing()` and the Langfuse SDK for full observability

**Next:** In Notebook 03, we'll run a systematic evaluation using the DeepSearchQA benchmark.

In [13]:
console.print(
    Panel(
        "[green]✓[/green] Notebook complete!\n\n"
        "[cyan]Next:[/cyan] Open [bold]03_evaluation.ipynb[/bold] to evaluate the agent at scale.",
        title="Done",
        border_style="green",
    )
)

╭────────────────────────────────────────────── Done ──────────────────────────────────────────────╮
│ ✓ Notebook complete!                                                                             │
│                                                                                                  │
│ Next: Open 03_evaluation.ipynb to evaluate the agent at scale.                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯